# Inspect Recordings Conversion

Use this notebook to sanity-check one raw `recordings` episode against its converted `lerobot_bi_yams` zarr episode. It shows the three camera streams before/after conversion, timestamp alignment, frame drops, zarr schema, and 14D joint traces.

Run it with the project environment, for example the `model/.venv` Python kernel, so `zarr`, `cv2`, `numpy`, `pandas`, and `matplotlib` are available.

Set `RAW_EPISODE_DIR` and `CONVERTED_ZARR_PATH` in the first code cell, then run top to bottom.

In [ ]:
from pathlib import Path

# Point these at one episode before and after conversion, or leave as None to auto-discover.
RAW_EPISODE_DIR = None
CONVERTED_ZARR_PATH = "data/teleop_converted/episode_0000.zarr"

# Matching the converter plan.
MAX_SYNC_MS = 50.0
TARGET_FRAME_SIZE = (640, 480)  # width, height
FRAME_INDICES_TO_SHOW = None  # None => start/middle/end, or set e.g. [0, 30, 60]

CAMERAS = {
    "workspace_rgb": {
        "raw_video": "camera_top-images-rgb.mp4",
        "raw_timestamps": "camera_top-rgb-timestamp.npy",
        "title": "top / workspace",
    },
    "wrist_rgb_left": {
        "raw_video": "camera_left-images-rgb.mp4",
        "raw_timestamps": "camera_left-rgb-timestamp.npy",
        "title": "left wrist",
    },
    "wrist_rgb_right": {
        "raw_video": "camera_right-images-rgb.mp4",
        "raw_timestamps": "camera_right-rgb-timestamp.npy",
        "title": "right wrist",
    },
}

print("configured raw episode:", RAW_EPISODE_DIR)
print("configured converted zarr:", CONVERTED_ZARR_PATH)

## Imports And Helpers

In [ ]:
import json
import math
import os
import sys
from dataclasses import dataclass
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr
from IPython.display import HTML, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import cv2
except Exception:
    cv2 = None

try:
    import decord
except Exception:
    decord = None

try:
    from data_preprocessing.action import process_recordings as converter
except Exception as exc:
    converter = None
    print("converter helper import unavailable; using notebook fallbacks:", repr(exc))

NS_PER_SEC = 1_000_000_000


def _decode_bytes(value: Any) -> str:
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, np.bytes_):
        return bytes(value).decode("utf-8", errors="replace")
    return str(value)


def _timestamp_seconds(ts: np.ndarray) -> np.ndarray:
    ts = np.asarray(ts)
    if ts.size == 0:
        return ts.astype(np.float64)
    ts_float = ts.astype(np.float64)
    # Converted zarr timestamps are uint64 nanoseconds relative to episode start.
    if np.issubdtype(ts.dtype, np.integer) and np.nanmax(ts_float) > 1e6:
        return ts_float / NS_PER_SEC
    return ts_float


def _nearest_indices(source_ts_sec: np.ndarray, target_ts_sec: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    source = np.asarray(source_ts_sec, dtype=np.float64)
    target = np.asarray(target_ts_sec, dtype=np.float64)
    if source.size == 0 or target.size == 0:
        return np.zeros(target.shape, dtype=np.int64), np.full(target.shape, np.nan, dtype=np.float64)
    right = np.searchsorted(source, target, side="left")
    left = np.clip(right - 1, 0, len(source) - 1)
    right = np.clip(right, 0, len(source) - 1)
    use_right = np.abs(source[right] - target) < np.abs(source[left] - target)
    idx = np.where(use_right, right, left)
    err_ms = np.abs(source[idx] - target) * 1000.0
    return idx.astype(np.int64), err_ms


def _resize_for_converter(frame: np.ndarray, target_size: tuple[int, int] = TARGET_FRAME_SIZE) -> np.ndarray:
    if converter is not None and hasattr(converter, "_resize_rgb_frame"):
        return converter._resize_rgb_frame(frame, target_size)
    if cv2 is None:
        return frame
    width, height = target_size
    if frame.shape[:2] == (height, width):
        return frame
    h, w = frame.shape[:2]
    target_aspect = width / height
    aspect = w / h
    if not math.isclose(aspect, target_aspect, rel_tol=1e-3):
        if aspect > target_aspect:
            new_w = int(round(h * target_aspect))
            x0 = max((w - new_w) // 2, 0)
            frame = frame[:, x0 : x0 + new_w]
        else:
            new_h = int(round(w / target_aspect))
            y0 = max((h - new_h) // 2, 0)
            frame = frame[y0 : y0 + new_h, :]
    return cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)


def decode_video_frame(video_path: Path, index: int) -> np.ndarray:
    video_path = Path(video_path)
    if decord is not None:
        vr = decord.VideoReader(str(video_path), ctx=decord.cpu(0), num_threads=1)
        index = int(np.clip(index, 0, len(vr) - 1))
        return vr[index].asnumpy()
    if cv2 is None:
        raise RuntimeError("Install either decord or opencv-python to decode raw mp4 frames in this notebook.")
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f"could not decode frame {index} from {video_path}")
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def video_frame_count(video_path: Path) -> int | None:
    try:
        if decord is not None:
            return len(decord.VideoReader(str(video_path), ctx=decord.cpu(0), num_threads=1))
        if cv2 is not None:
            cap = cv2.VideoCapture(str(video_path))
            n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            return n
    except Exception:
        return None
    return None


def read_raw_camera_timestamps(raw_episode_dir: Path, cam_cfg: dict[str, str]) -> np.ndarray:
    if converter is not None and hasattr(converter, "load_camera_timestamps"):
        # The converter helper signature may be either (episode_dir, camera) or direct path in early drafts.
        try:
            camera_name = cam_cfg["raw_timestamps"].removeprefix("camera_").removesuffix("-rgb-timestamp.npy")
            return np.asarray(converter.load_camera_timestamps(raw_episode_dir, camera_name))
        except Exception:
            pass
    return np.load(raw_episode_dir / cam_cfg["raw_timestamps"])


def summarize_zarr(root: zarr.Group) -> pd.DataFrame:
    rows = []
    for key in sorted(root.keys()):
        arr = root[key]
        rows.append({
            "key": key,
            "shape": tuple(arr.shape),
            "dtype": str(arr.dtype),
            "chunks": tuple(arr.chunks) if hasattr(arr, "chunks") else None,
        })
    return pd.DataFrame(rows)


def _session_joint_topic(raw_episode_dir: Path, side: str) -> str:
    meta_path = raw_episode_dir / "session_meta.json"
    if not meta_path.exists():
        return "joint_state"
    meta = json.loads(meta_path.read_text())
    candidates = [
        (side, "joint_state"),
        (f"yam_{side}", "joint_state"),
        ("topics", f"yam_{side}", "joint_state"),
        ("topics", side, "joint_state"),
    ]
    for path in candidates:
        cur = meta
        for part in path:
            if isinstance(cur, dict) and part in cur:
                cur = cur[part]
            else:
                cur = None
                break
        if isinstance(cur, str):
            return cur
    for key, value in meta.items() if isinstance(meta, dict) else []:
        if side in str(key) and "joint" in str(key) and isinstance(value, str):
            return value
    return "joint_state"


def load_raw_yam_joints(raw_episode_dir: Path) -> tuple[np.ndarray, np.ndarray] | None:
    """Return raw MCAP joints as (timestamps_sec, joints_14d) when converter helpers are available."""
    if converter is None or not hasattr(converter, "read_mcap_joint_stream"):
        print("Raw MCAP comparison skipped: converter.read_mcap_joint_stream is not importable yet.")
        return None
    streams = []
    for side in ("left", "right"):
        mcap_path = raw_episode_dir / f"yam_{side}.mcap"
        if not mcap_path.exists():
            print(f"Raw MCAP comparison skipped: missing {mcap_path}")
            return None
        topic = _session_joint_topic(raw_episode_dir, side)
        ts, values = converter.read_mcap_joint_stream(mcap_path, topic)
        streams.append((_timestamp_seconds(np.asarray(ts)), np.asarray(values, dtype=np.float32)))
    left_ts, left = streams[0]
    right_ts, right = streams[1]
    anchor = anchor_z if "anchor_z" in globals() else left_ts
    left_idx, _ = _nearest_indices(left_ts - raw_ts_sec[anchor_key][0], anchor)
    right_idx, _ = _nearest_indices(right_ts - raw_ts_sec[anchor_key][0], anchor)
    return anchor, np.concatenate([left[left_idx], right[right_idx]], axis=1)

## Load Episode And Check Schema

In [ ]:
def _raw_episode_candidates() -> list[Path]:
    bases = [
        REPO_ROOT / "data" / "teleop_raw",
        Path.home() / "Downloads" / "recordings",
    ]
    candidates = []
    for base in bases:
        if base.exists():
            candidates.extend(p for p in base.glob("*/*") if p.is_dir() and p.name.startswith("episode"))
    return sorted(set(candidates))


def _episode_index_from_zarr_name(path: Path) -> int | None:
    if not path.name.startswith("episode_") or path.suffix != ".zarr":
        return None
    try:
        return int(path.stem.removeprefix("episode_"))
    except ValueError:
        return None


def _normalize_path(path: Path) -> Path:
    path = Path(path).expanduser()
    if path.exists() or path.is_absolute():
        return path
    repo_relative = REPO_ROOT / path
    return repo_relative if repo_relative.exists() else path


def _converted_counterpart(path: Path) -> Path:
    if path.exists():
        return path
    candidates = [
        REPO_ROOT / "data" / "teleop_converted" / path.name,
        Path.home() / "Downloads" / path.name,
        Path.home() / "zarr" / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return path


def _source_episode_from_zarr(path: Path) -> Path | None:
    if not path.exists() or path.suffix != ".zarr":
        return None
    try:
        root = zarr.open(str(path), mode="r")
        source = root.attrs.get("source_episode") or root.attrs.get("source_episode_path")
    except Exception:
        return None
    if not source:
        return None
    source_path = Path(source).expanduser()
    if not source_path.is_absolute():
        source_path = (REPO_ROOT / source_path).resolve()
    return source_path


def _converted_for_source_episode_name(source_episode_name: str) -> Path | None:
    search_dirs = [REPO_ROOT / "data" / "teleop_converted", Path.home() / "Downloads", Path.home() / "zarr"]
    for base in search_dirs:
        if not base.exists():
            continue
        for candidate in sorted(base.glob("episode_*.zarr")):
            source = _source_episode_from_zarr(candidate)
            if source is not None and source.name == source_episode_name:
                return candidate
    return None


def discover_raw_episode(preferred_zarr: Path | None = None) -> Path:
    if preferred_zarr is not None:
        source = _source_episode_from_zarr(preferred_zarr)
        if source is not None:
            return source
        idx = _episode_index_from_zarr_name(preferred_zarr)
        candidates = _raw_episode_candidates()
        if idx is not None and idx < len(candidates):
            return candidates[idx]

    candidates = _raw_episode_candidates()
    if not candidates:
        raise FileNotFoundError("No raw episode directories found under data/teleop_raw or ~/Downloads/recordings")
    return candidates[0]


def discover_converted_zarr() -> Path:
    search_dirs = [REPO_ROOT / "data" / "teleop_converted", Path.home() / "Downloads", Path.home() / "zarr", REPO_ROOT]
    candidates = []
    for base in search_dirs:
        if base.exists():
            candidates.extend(sorted(base.glob("episode_*.zarr")))
            candidates.extend(sorted(base.glob("**/episode_*.zarr")) if base == REPO_ROOT else [])
    candidates = [p for p in candidates if p.is_dir()]
    if not candidates:
        raise FileNotFoundError("No converted episode_*.zarr directory found in ~/Downloads, ~/zarr, or the repo")
    return candidates[0]


RAW_EPISODE_DIR = _normalize_path(Path(RAW_EPISODE_DIR)) if RAW_EPISODE_DIR is not None else None
CONVERTED_ZARR_PATH = _normalize_path(Path(CONVERTED_ZARR_PATH)) if CONVERTED_ZARR_PATH is not None else None

# If a converted zarr was accidentally entered as RAW_EPISODE_DIR, treat it as the converted path.
if RAW_EPISODE_DIR is not None and RAW_EPISODE_DIR.suffix == ".zarr":
    if CONVERTED_ZARR_PATH is None:
        CONVERTED_ZARR_PATH = _converted_counterpart(RAW_EPISODE_DIR)
    RAW_EPISODE_DIR = None

if CONVERTED_ZARR_PATH is not None and not CONVERTED_ZARR_PATH.exists():
    corrected = _converted_counterpart(CONVERTED_ZARR_PATH)
    if not corrected.exists():
        corrected = _converted_for_source_episode_name(CONVERTED_ZARR_PATH.name) or corrected
    CONVERTED_ZARR_PATH = corrected

CONVERTED_ZARR_PATH = CONVERTED_ZARR_PATH if CONVERTED_ZARR_PATH is not None else discover_converted_zarr()
RAW_EPISODE_DIR = RAW_EPISODE_DIR if RAW_EPISODE_DIR is not None else discover_raw_episode(CONVERTED_ZARR_PATH)

assert RAW_EPISODE_DIR.exists(), f"missing raw episode: {RAW_EPISODE_DIR}"
assert CONVERTED_ZARR_PATH.exists(), f"missing converted zarr: {CONVERTED_ZARR_PATH}"
print("raw episode:", RAW_EPISODE_DIR)
print("converted zarr:", CONVERTED_ZARR_PATH)


raw_ts = {key: read_raw_camera_timestamps(RAW_EPISODE_DIR, cfg) for key, cfg in CAMERAS.items()}
raw_ts_sec = {key: _timestamp_seconds(ts) for key, ts in raw_ts.items()}
z = zarr.open(str(CONVERTED_ZARR_PATH), mode="r")

display(summarize_zarr(z))
print("zarr attrs:")
print(json.dumps(dict(z.attrs), indent=2, default=str))

if "language_instruction" in z:
    print("language_instruction:", _decode_bytes(z["language_instruction"][0]))

In [ ]:
summary_rows = []
for key, cfg in CAMERAS.items():
    video_path = RAW_EPISODE_DIR / cfg["raw_video"]
    z_ts_key = f"{key}_timestamps"
    z_len = z[key].shape[0] if key in z else None
    z_ts = z[z_ts_key][:] if z_ts_key in z else np.array([])
    summary_rows.append({
        "stream": key,
        "raw_video_exists": video_path.exists(),
        "raw_video_frames": video_frame_count(video_path) if video_path.exists() else None,
        "raw_timestamp_count": len(raw_ts[key]),
        "raw_duration_s": float(raw_ts_sec[key][-1] - raw_ts_sec[key][0]) if len(raw_ts_sec[key]) > 1 else np.nan,
        "zarr_shape": tuple(z[key].shape) if key in z else None,
        "zarr_timestamp_count": len(z_ts),
        "zarr_dtype": str(z[key].dtype) if key in z else None,
        "zarr_duration_s": float(_timestamp_seconds(z_ts)[-1] - _timestamp_seconds(z_ts)[0]) if len(z_ts) > 1 else np.nan,
    })

if "joint_state_lowdim" in z:
    summary_rows.append({
        "stream": "joint_state_lowdim",
        "raw_video_exists": None,
        "raw_video_frames": None,
        "raw_timestamp_count": None,
        "raw_duration_s": None,
        "zarr_shape": tuple(z["joint_state_lowdim"].shape),
        "zarr_timestamp_count": len(z["joint_state_lowdim_timestamps"][:]) if "joint_state_lowdim_timestamps" in z else None,
        "zarr_dtype": str(z["joint_state_lowdim"].dtype),
        "zarr_duration_s": None,
    })

summary = pd.DataFrame(summary_rows)
display(summary)

## Alignment Diagnostics

The converter plan uses top-camera timestamps as the anchor and nearest-neighbor matching for left/right camera frames and joint states. This section checks the camera side of that contract.

In [ ]:
anchor_key = "workspace_rgb"
anchor_raw = raw_ts_sec[anchor_key] - raw_ts_sec[anchor_key][0]
anchor_z = _timestamp_seconds(z[f"{anchor_key}_timestamps"][:]) if f"{anchor_key}_timestamps" in z else anchor_raw

diag_rows = []
nearest_by_camera = {}
for key in CAMERAS:
    source = raw_ts_sec[key] - raw_ts_sec[anchor_key][0]
    idx, err_ms = _nearest_indices(source, anchor_z)
    nearest_by_camera[key] = (idx, err_ms)
    diag_rows.append({
        "camera": key,
        "raw_count": len(source),
        "zarr_count": z[key].shape[0] if key in z else np.nan,
        "nearest_error_mean_ms": float(np.nanmean(err_ms)),
        "nearest_error_p95_ms": float(np.nanpercentile(err_ms, 95)),
        "nearest_error_max_ms": float(np.nanmax(err_ms)),
        "frames_over_threshold": int(np.sum(err_ms > MAX_SYNC_MS)),
    })

diag = pd.DataFrame(diag_rows)
display(diag)

fig, ax = plt.subplots(figsize=(10, 4))
for key, (_idx, err_ms) in nearest_by_camera.items():
    ax.plot(err_ms, label=key, linewidth=1.5)
ax.axhline(MAX_SYNC_MS, color="crimson", linestyle="--", linewidth=1, label=f"{MAX_SYNC_MS:g} ms threshold")
ax.set_title("Nearest raw camera timestamp error vs converted anchor frame")
ax.set_xlabel("converted frame index")
ax.set_ylabel("absolute timestamp error (ms)")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.25)
plt.show()

## Before/After Frame Comparison

In [ ]:
def _default_frame_indices(T: int) -> list[int]:
    if T <= 0:
        return []
    return sorted(set([0, T // 2, T - 1]))


T = min(z[key].shape[0] for key in CAMERAS if key in z)
frame_indices = FRAME_INDICES_TO_SHOW if FRAME_INDICES_TO_SHOW is not None else _default_frame_indices(T)
frame_indices = [int(np.clip(i, 0, T - 1)) for i in frame_indices]
print("showing converted indices:", frame_indices)

for t in frame_indices:
    fig, axes = plt.subplots(len(CAMERAS), 3, figsize=(13, 4 * len(CAMERAS)))
    if len(CAMERAS) == 1:
        axes = axes[None, :]
    fig.suptitle(f"Converted frame {t} / {T - 1}", fontsize=14)

    for row, (key, cfg) in enumerate(CAMERAS.items()):
        raw_idx = int(nearest_by_camera[key][0][t]) if key in nearest_by_camera else t
        raw_frame = decode_video_frame(RAW_EPISODE_DIR / cfg["raw_video"], raw_idx)
        raw_resized = _resize_for_converter(raw_frame)
        converted = np.asarray(z[key][t])
        diff = np.abs(raw_resized.astype(np.int16) - converted.astype(np.int16)).astype(np.uint8)

        axes[row, 0].imshow(raw_resized)
        axes[row, 0].set_title(f"raw nearest {cfg['title']}\nraw index={raw_idx}")
        axes[row, 1].imshow(converted)
        axes[row, 1].set_title(f"converted {key}\nzarr index={t}")
        axes[row, 2].imshow(diff)
        axes[row, 2].set_title(f"abs pixel diff\nmean={diff.mean():.2f}, max={diff.max()}")
        for ax in axes[row]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

## Interactive Frame Scrubber

In [ ]:
try:
    import ipywidgets as widgets
except Exception as exc:
    widgets = None
    print("ipywidgets unavailable; install it for the scrubber:", repr(exc))


def show_frame(t: int = 0):
    t = int(np.clip(t, 0, T - 1))
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    fig.suptitle(f"zarr frame {t}; timestamp={anchor_z[t]:.3f}s")
    for col, (key, cfg) in enumerate(CAMERAS.items()):
        raw_idx = int(nearest_by_camera[key][0][t])
        raw_frame = _resize_for_converter(decode_video_frame(RAW_EPISODE_DIR / cfg["raw_video"], raw_idx))
        converted = np.asarray(z[key][t])
        axes[0, col].imshow(raw_frame)
        axes[0, col].set_title(f"raw {cfg['title']} #{raw_idx}")
        axes[1, col].imshow(converted)
        axes[1, col].set_title(f"converted {key} #{t}")
        axes[0, col].axis("off")
        axes[1, col].axis("off")
    plt.tight_layout()
    plt.show()


if widgets is not None:
    display(widgets.interact(show_frame, t=widgets.IntSlider(min=0, max=max(T - 1, 0), step=1, value=0)))
else:
    show_frame(0)

## Raw MCAP Joint Comparison

This cell uses `data_preprocessing.action.process_recordings.read_mcap_joint_stream` when that helper exists. If the other agent has not landed it yet, the notebook keeps going and the converted joint plots below still run.

In [ ]:
raw_joint_result = load_raw_yam_joints(RAW_EPISODE_DIR)
raw_joints_aligned = None
if raw_joint_result is not None:
    raw_joint_ts, raw_joints_aligned = raw_joint_result
    converted_joints_for_compare = np.asarray(z["joint_state_lowdim"][: len(raw_joints_aligned)])
    joint_diff = raw_joints_aligned[: len(converted_joints_for_compare)] - converted_joints_for_compare
    display(pd.DataFrame({
        "metric": ["mean_abs", "p95_abs", "max_abs"],
        "value": [
            float(np.mean(np.abs(joint_diff))),
            float(np.percentile(np.abs(joint_diff), 95)),
            float(np.max(np.abs(joint_diff))),
        ],
    }))

    dims_to_show = list(range(min(14, raw_joints_aligned.shape[1])))
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    for dim in dims_to_show[:7]:
        axes[0].plot(raw_joint_ts, raw_joints_aligned[:, dim], linestyle="--", linewidth=1, label=f"raw L{dim}")
        axes[0].plot(anchor_z[: len(converted_joints_for_compare)], converted_joints_for_compare[:, dim], linewidth=1, label=f"zarr L{dim}")
    for dim in dims_to_show[7:14]:
        axes[1].plot(raw_joint_ts, raw_joints_aligned[:, dim], linestyle="--", linewidth=1, label=f"raw R{dim - 7}")
        axes[1].plot(anchor_z[: len(converted_joints_for_compare)], converted_joints_for_compare[:, dim], linewidth=1, label=f"zarr R{dim - 7}")
    axes[0].set_title("raw MCAP vs converted zarr joints: left")
    axes[1].set_title("raw MCAP vs converted zarr joints: right")
    axes[1].set_xlabel("timestamp (s)")
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend(ncol=4, fontsize=7)
    plt.tight_layout()
    plt.show()

## Joint State Checks

In [ ]:
assert "joint_state_lowdim" in z, "converted zarr is missing joint_state_lowdim"
joints = np.asarray(z["joint_state_lowdim"][:])
joint_ts = _timestamp_seconds(z["joint_state_lowdim_timestamps"][:]) if "joint_state_lowdim_timestamps" in z else anchor_z

print("joint shape:", joints.shape, "dtype:", joints.dtype)
print("finite:", bool(np.isfinite(joints).all()))
print("left arm dims 0:7, right arm dims 7:14")

joint_summary = pd.DataFrame({
    "dim": np.arange(joints.shape[1]),
    "side": ["left" if i < 7 else "right" for i in range(joints.shape[1])],
    "min": joints.min(axis=0),
    "max": joints.max(axis=0),
    "mean": joints.mean(axis=0),
    "std": joints.std(axis=0),
})
display(joint_summary)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for i in range(min(7, joints.shape[1])):
    axes[0].plot(joint_ts, joints[:, i], label=f"L{i}", linewidth=1)
for i in range(7, min(14, joints.shape[1])):
    axes[1].plot(joint_ts, joints[:, i], label=f"R{i - 7}", linewidth=1)
axes[0].set_title("left YAM joint_state_lowdim dims 0-6")
axes[1].set_title("right YAM joint_state_lowdim dims 7-13")
axes[1].set_xlabel("converted timestamp (s)")
for ax in axes:
    ax.set_ylabel("joint value")
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=7, fontsize=8)
plt.tight_layout()
plt.show()

## Hard Validation Summary

In [ ]:
checks = []

def check(name: str, ok: bool, detail: str = ""):
    checks.append({"check": name, "ok": bool(ok), "detail": detail})


lengths = {}
for key in CAMERAS:
    lengths[key] = z[key].shape[0]
    lengths[f"{key}_timestamps"] = z[f"{key}_timestamps"].shape[0]
lengths["joint_state_lowdim"] = z["joint_state_lowdim"].shape[0]
lengths["joint_state_lowdim_timestamps"] = z["joint_state_lowdim_timestamps"].shape[0]

check("all aligned arrays share T", len(set(lengths.values())) == 1, str(lengths))
for key in CAMERAS:
    arr = z[key]
    check(f"{key} shape is (T,480,640,3)", arr.ndim == 4 and arr.shape[1:] == (480, 640, 3), str(arr.shape))
    check(f"{key} dtype uint8", arr.dtype == np.uint8, str(arr.dtype))
    ts = z[f"{key}_timestamps"][:]
    check(f"{key} timestamps strictly increasing", bool(np.all(np.diff(ts) > 0)), f"count={len(ts)}")

check("joint_state_lowdim shape is (T,14)", joints.ndim == 2 and joints.shape[1] == 14, str(joints.shape))
check("joint_state_lowdim dtype float32", joints.dtype == np.float32, str(joints.dtype))
check("joint_state_lowdim finite", bool(np.isfinite(joints).all()))
check("language_instruction exists", "language_instruction" in z)

result = pd.DataFrame(checks)
display(result)
if not result["ok"].all():
    raise AssertionError("One or more conversion validation checks failed; inspect the table above.")
print("All hard validation checks passed.")